# AIC2024 Phase 1 - index tren Colab

Pipeline: TransNetV2 -> shot -> keyframe -> SQLite/JPEG.

**Truoc khi chay:** Runtime -> Change runtime type -> GPU (T4).
Upload/clone repo vao `/content/IR`.

Notebook cai package vao system Python cua Colab (`uv pip install --system`) de dung torch CUDA co san, khong tao venv Python 3.11 rieng (venv do se khong thay torch cua Colab).

In [ ]:
!nvidia-smi
import sys, torch
print(f"python={sys.version.split()[0]} torch={torch.__version__} cuda={torch.cuda.is_available()}")

In [ ]:
!pip install -q uv
!uv --version

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Tuy chon: restore artifacts cu de resume (bo qua neu bat dau moi)
from pathlib import Path

RESTORE = Path("/content/drive/MyDrive/aic2024/artifacts.zip")
if RESTORE.exists():
    !cd /content/IR && unzip -n "{RESTORE}"
    print("Da restore artifacts tu Drive")
else:
    print("Chua co artifacts.zip, bat dau index moi")

In [ ]:
%cd /content/IR
!uv pip install --system -e .
!uv pip install --system --group ml --group index
import av, torch, transnetv2_pytorch
print("torch", torch.__version__, "cuda", torch.cuda.is_available())

In [ ]:
from pathlib import Path

REPO = Path("/content/IR")
CORPUS = Path("/content/drive/MyDrive/aic2024/videos")
ARTIFACTS_ZIP = Path("/content/drive/MyDrive/aic2024/artifacts.zip")

assert REPO.exists(), REPO
assert CORPUS.exists(), f"Khong thay corpus: {CORPUS}"
videos = list(CORPUS.rglob("*.mp4"))
print(f"corpus={CORPUS} videos={len(videos)}")

In [ ]:
# Chay thu 1 video truoc khi index full
!ir index --corpus "{CORPUS}" --limit 1
!ir stats

In [ ]:
# Index full corpus (co the mat nhieu gio; pipeline resume theo video/step)
!ir index --corpus "{CORPUS}"
!ir stats

In [ ]:
# Zip artifacts ve Drive de dung cho Phase 2 / resume lan sau
ARTIFACTS_ZIP.parent.mkdir(parents=True, exist_ok=True)
!cd /content/IR && zip -qr -FS "{ARTIFACTS_ZIP}" artifacts
print(f"Da zip artifacts: {ARTIFACTS_ZIP} ({ARTIFACTS_ZIP.stat().st_size / 1024 / 1024:.1f} MB)")

## Ghi chu

- Neu session bi ngat: chay lai cell restore (neu da zip) roi chay lai `ir index`; video da xong se duoc skip.
- `artifacts/` nam trong runtime Colab, phai zip ve Drive truoc khi het phien.
- Chi index shot: `ir index --only shots`; index lai mot video: `ir index --reindex --limit ...` (can nhac vi reindex toan bo).
- Phase 2 can them group `serve` (FAISS + FastAPI): `uv pip install --system --group ml --group index --group serve`.